# Open Data Event Displays

Example event displays from the runs 1-5 BNB open data for each of the 20 BAGELS multiclass-BDT selection channels.
The internal note section "Open Data Event Displays" is built from this notebook.

* **Selections.** The channel cuts and orthogonality priorities are copied from `open_data_distributions.ipynb`. Keep the
  `bdt_cut_thresholds` and priorities below in sync with that notebook. They apply on top of the generic neutrino preselection
  (`wc_kine_reco_Enu > 0`).
* **Events.** One hand-picked open-data event per channel, stored as run/subrun/event in `display_events` below. They were chosen from
  random draws that passed the selections of the `all_vars_r16_2026_10_07` BDT. If a stored event no longer passes its channel's selection
  (e.g. after a BDT retrain), the notebook prints a loud warning but still draws it; pick a new event for that channel in that case.
* **Displays.** These show the Wire-Cell `rec_charge_blob` spacepoints (`wcpselection/T_spacepoints`, `Trecchargeblob_spacepoints_*`),
  the reconstructed 3D charge blobs of the in-beam (neutrino-candidate) cluster.
  * The view is $x$-$z$: $z$ runs along the beam (horizontal) and $x$ is the drift coordinate (vertical). Points are colored by charge with the jet color map,
    from 0 to the event's 99th-percentile charge.
  * The axes share one scale. The window covers the 5th-95th percentile range of the spacepoints in $x$ and in $z$. It is expanded if needed to include the
    reco vertex, padded by `PAD_CM`, and then widened in the shorter direction so that every display has the same aspect ratio.
  * Black lines mark the TPC active-volume boundaries ($-1 < x < 254.3$ cm, $0.6 < z < 1036.4$ cm) where they fall inside the window.
  * The magenta star is the Wire-Cell reco neutrino vertex (`wc_reco_nuvtxX/Z`).
  * The box on the right lists a few reco quantities and the channel's BDT score.
* **Overlays.** A second set of plots shows the whole detector for each channel, with the `rec_charge_blob` spacepoints of up to
  `max_overlay_events` selected open-data events overlaid, one color per event (the color cycle repeats once it runs out). If a channel has more events,
  a fixed-seed random subset is shown. The main panel is the $z$-$x$ view, with the $z$-$y$ view above it and the $y$-$x$ view to its right;
  the panels share their axes. Each event's reco vertex is drawn as a small dot in its color.
* **Matching.** The ROOT trees have no event index in `all_df`, so events are matched to `wcpselection/T_eval` entries by run/subrun/event.
  `T_spacepoints` has no run/subrun/event branches; it has the same number of entries as `T_eval` and is read by entry number.
* **Output.** Each display is saved as PNG and PDF under `plots/open_data_event_displays/`, and each overlay under `plots/open_data_event_displays/overlays/`. The LaTeX subsection is written to
  `plots/open_data_event_displays/open_data_event_displays.tex`. Its figure paths use `\\odedfigs`, which the note defines.

In [ ]:
import os, sys, warnings
import numpy as np
import polars as pl
import awkward as ak
import uproot
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 110

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.file_locations import intermediate_files_location, data_files_location
from src.signal_categories import train_category_labels, train_category_labels_latex

training = "all_vars_r16_2026_10_07"
reco_categories = train_category_labels
reco_category_labels_latex = train_category_labels_latex
prob_categories = ["prob_" + cat for cat in reco_categories]

# one hand-picked display event per channel, (run, subrun, event); chosen with the all_vars_r16_2026_10_07 BDT selections
display_events = {
    "1gNp":             (5184, 53, 2673),
    "1g0p":             (20838, 106, 5314),
    "1gNp1mu":          (21021, 111, 5563),
    "1g0p1mu":          (21248, 13, 662),
    "1g_outFV":         (5384, 130, 6502),
    "NC1pi0_Np":        (21279, 24, 1239),
    "NC1pi0_0p":        (5906, 63, 3188),
    "numuCC1pi0_Np":    (19509, 479, 23962),
    "numuCC1pi0_0p":    (16122, 332, 16644),
    "1pi0_outFV":       (21157, 245, 12255),
    "nueCC_Np":         (20770, 114, 5730),
    "nueCC_0p":         (20700, 229, 11466),
    "numuCC_Np":        (19545, 19, 979),
    "numuCC_0p":        (20748, 110, 5510),
    "other_outFV_dirt": (14335, 20, 1039),
    "multi_pi0":        (19708, 167, 8385),
    "eta_other":        (19493, 88, 4419),
    "pi0_dalitz_decay": (5399, 14, 716),
    "NC_no_gamma":      (20939, 280, 14026),
    "ext":              (16453, 32, 1626),
}
assert sorted(display_events) == sorted(reco_categories)

PAD_CM = 30.0
TPC_X, TPC_Y, TPC_Z = (-1.0, 254.3), (-115.0, 117.0), (0.6, 1036.4)   # TPC active volume (cm), as in src/blip_postprocessing.py

max_overlay_events = 200
overlay_seed = 20261008

# With show_plots False the figures are still saved but not drawn inline
show_plots = True

plots_base = "open_data_event_displays"
plots_dir = f"../plots/{plots_base}"
os.makedirs(plots_dir, exist_ok=True)

# Loading The Open Data

All data rows are loaded and left-joined with the BDT predictions, which exist only for events that pass the training preselection. Events without a
prediction get BDT scores of -1, as in `open_data_distributions.ipynb`. The left join keeps them, so a display event that fails the preselection
can still be found and flagged below.

In [ ]:
info_vars = ["detailed_run_period", "wc_kine_reco_Enu", "wc_reco_nuvtxX", "wc_reco_nuvtxY", "wc_reco_nuvtxZ",
             "wc_reco_showerKE", "wc_reco_muonMomentum_3", "wc_reco_num_protons_35_MeV", "wc_kine_pio_mass"]

data_df = (
    pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")
    .filter(pl.col("filetype") == "data")
    .select(["filename", "filetype", "run", "subrun", "event"] + info_vars)
)
pred_df = pl.scan_parquet(f"../training_outputs/{training}/predictions.parquet").filter(pl.col("filetype") == "data")
data_df = (
    data_df.join(pred_df.select(["filetype", "run", "subrun", "event"] + prob_categories),
                 on=["filetype", "run", "subrun", "event"], how="left")
    .collect()
    .with_columns([pl.col(prob).fill_null(-1) for prob in prob_categories])
)
print(f"{data_df.height} open-data events, {data_df.filter(pl.col('wc_kine_reco_Enu') > 0).height} pass the generic preselection")
print(data_df.group_by("filename").len().sort("filename"))

# BDT Reco Category Selections

Same cuts and orthogonality priorities as `open_data_distributions.ipynb`. Categories with no tuned cut fall back to the argmax of the BDT probabilities.

In [ ]:
data_df = data_df.with_columns(pl.concat_list(prob_categories).list.arg_max().alias("reco_category_argmax_index"))

bdt_cut_thresholds = {
    "1gNp": 0.3, "1g0p": 0.9, "1gNp1mu": 0.5, "1g0p1mu": 0.2, "1g_outFV": 0.5,
    "NC1pi0_Np": None, "NC1pi0_0p": None,
    "numuCC1pi0_Np": 0.1, "numuCC1pi0_0p": 0.15, "1pi0_outFV": 0.1,
    "nueCC_Np": 0.05, "nueCC_0p": 0.05,
    "numuCC_Np": 0.5, "numuCC_0p": 0.5,
    "other_outFV_dirt": None,
    "multi_pi0": 0.02, "eta_other": 0.01, "pi0_dalitz_decay": 0.1,
    "NC_no_gamma": None, "ext": None,
}

# orthogonality priority (lower wins)
selection_priorities = {
    "1gNp": 1, "1g0p": 2, "1gNp1mu": 3, "1g0p1mu": 4, "1g_outFV": 5,
    "NC1pi0_Np": 6, "NC1pi0_0p": 7,
    "numuCC1pi0_Np": 9, "numuCC1pi0_0p": 8,   # 0p takes priority over Np in orthogonality
    "1pi0_outFV": 10,
    "nueCC_Np": 12, "nueCC_0p": 11,           # 0p takes priority over Np in orthogonality
    "numuCC_Np": 13, "numuCC_0p": 14,
    "other_outFV_dirt": 15, "multi_pi0": 16, "eta_other": 17,
    "pi0_dalitz_decay": 5.5,                  # high priority for dalitz, rare topology
    "NC_no_gamma": 19, "ext": 20,
}

def raw_selection(i, name):
    thr = bdt_cut_thresholds[name]
    return pl.col("reco_category_argmax_index") == i if thr is None else pl.col(f"prob_{name}") > thr

raw_queries = {name: raw_selection(i, name) for i, name in enumerate(reco_categories)}
by_priority = sorted(reco_categories, key=lambda name: selection_priorities[name])
reco_category_query_dic = {}
for k, name in enumerate(by_priority):
    expr = raw_queries[name]
    for higher in by_priority[:k]:
        expr = expr & ~raw_queries[higher]
    reco_category_query_dic[name] = expr

data_df = data_df.with_columns(
    pl.when(pl.col("wc_kine_reco_Enu") > 0)
    .then(pl.coalesce([pl.when(reco_category_query_dic[name]).then(pl.lit(name)) for name in reco_categories]))
    .alias("reco_channel")
)
counts = dict(data_df.group_by("reco_channel").len().iter_rows())
for name in reco_categories:
    print(f"{name:>18s}: {counts.get(name, 0)} open-data events")

# Finding The Display Events

Each stored event is looked up by run/subrun/event (the open-data files cover disjoint runs, so this is unique) and checked against its channel's
selection. Then it is matched to its `T_eval` entry number.

In [ ]:
rse_rows = pl.DataFrame(
    [(channel, *rse) for channel, rse in display_events.items()],
    schema={"display_channel": pl.Utf8, "run": pl.Int32, "subrun": pl.Int32, "event": pl.Int32}, orient="row",
)
chosen_df = rse_rows.join(data_df, on=["run", "subrun", "event"], how="left")
assert chosen_df.height == len(display_events), "a display event matches more than one open-data row"

problems = []
for row in chosen_df.iter_rows(named=True):
    rse = f"{row['run']}/{row['subrun']}/{row['event']}"
    if row["filename"] is None:
        problems.append(f"{row['display_channel']}: event {rse} is not in the open data at all")
    elif row["reco_channel"] != row["display_channel"]:
        score = row[f"prob_{row['display_channel']}"]
        problems.append(f"{row['display_channel']}: event {rse} is now selected as {row['reco_channel']} "
                        f"(prob_{row['display_channel']} = {score:.3f})")
if problems:
    banner = "!" * 100
    msg = (f"\n{banner}\nWARNING: {len(problems)} DISPLAY EVENT(S) DO NOT PASS THEIR CHANNEL'S SELECTION WITH THE {training} BDT.\n"
           "Pick new events for these channels in display_events (the displays below are still drawn):\n  "
           + "\n  ".join(problems) + f"\n{banner}\n")
    print(msg)
    warnings.warn(msg)
else:
    print(f"all {len(display_events)} display events pass their channel's selection with the {training} BDT")

chosen_df = chosen_df.filter(pl.col("filename").is_not_null())
entries = np.full(chosen_df.height, -1)
for filename in chosen_df["filename"].unique():
    t_eval = uproot.open(f"{data_files_location}/{filename}")["wcpselection/T_eval"]
    rse = t_eval.arrays(["run", "subrun", "event"], library="np")
    entry_of = {k: i for i, k in enumerate(zip(rse["run"], rse["subrun"], rse["event"]))}
    assert len(entry_of) == t_eval.num_entries, f"duplicate run/subrun/event in {filename}"
    for row_i, (fn, r, s_, e) in enumerate(chosen_df.select(["filename", "run", "subrun", "event"]).iter_rows()):
        if fn == filename:
            entries[row_i] = entry_of[(r, s_, e)]
assert (entries >= 0).all()
chosen_df = chosen_df.with_columns(pl.Series("entry", entries))

# Event Display Function

In [ ]:
_files = {}

def read_spacepoints(filename, entry, coords=("x", "z", "q")):
    if filename not in _files:
        _files[filename] = uproot.open(f"{data_files_location}/{filename}")
    t_sp = _files[filename]["wcpselection/T_spacepoints"]
    branches = [f"Trecchargeblob_spacepoints_{c}" for c in coords]
    sp = t_sp.arrays(branches, entry_start=entry, entry_stop=entry + 1)
    return [ak.to_numpy(sp[b][0]) for b in branches]


def fmt(val, unit, scale=1.0, sentinel_below=0.0):
    return "none" if val is None or not np.isfinite(val) or val <= sentinel_below else f"{val * scale:.0f} {unit}"


def info_label(row, channel):
    lines = [
        f"run {row['run']} / {row['subrun']} / {row['event']}",
        f"run period {row['detailed_run_period']}",
        "",
        f"{channel} BDT score: {row[f'prob_{channel}']:.3f}",
        f"reco $E_\\nu$: {fmt(row['wc_kine_reco_Enu'], 'MeV')}",
        f"reco shower KE: {fmt(row['wc_reco_showerKE'], 'MeV', scale=1e3)}",
        f"reco muon E: {fmt(row['wc_reco_muonMomentum_3'], 'MeV', scale=1e3)}",
        f"reco protons (KE > 35 MeV): {row['wc_reco_num_protons_35_MeV']}",
        f"reco $\\gamma\\gamma$ mass: {fmt(row['wc_kine_pio_mass'], 'MeV')}",
    ]
    return "\n".join(lines)


FIG_W, FIG_H = 10.5, 4.6   # every display has the same size, so they tile neatly in the note
AX_RECT = [0.07, 0.13, 0.62, 0.75]   # main axes: [left, bottom, width, height] in figure fractions

def draw_event(row, channel, channel_latex):
    x, z, q = read_spacepoints(row["filename"], row["entry"])
    if len(x) == 0:
        print(f"no spacepoints for {channel} run {row['run']}/{row['subrun']}/{row['event']}, skipping")
        return None
    vtx = np.array([row["wc_reco_nuvtxX"], row["wc_reco_nuvtxY"], row["wc_reco_nuvtxZ"]])

    # 5th-95th percentile of the spacepoints, expanded to include the reco vertex, plus the padding
    x_lo, x_hi = np.percentile(x, [5, 95])
    z_lo, z_hi = np.percentile(z, [5, 95])
    x_lo, x_hi = min(x_lo, vtx[0]) - PAD_CM, max(x_hi, vtx[0]) + PAD_CM
    z_lo, z_hi = min(z_lo, vtx[2]) - PAD_CM, max(z_hi, vtx[2]) + PAD_CM
    # widen the shorter direction so the window matches the axes' aspect ratio (equal scale on both axes)
    target = (AX_RECT[3] * FIG_H) / (AX_RECT[2] * FIG_W)   # (x extent) / (z extent)
    if (x_hi - x_lo) / (z_hi - z_lo) < target:
        extra = target * (z_hi - z_lo) - (x_hi - x_lo)
        x_lo, x_hi = x_lo - extra / 2, x_hi + extra / 2
    else:
        extra = (x_hi - x_lo) / target - (z_hi - z_lo)
        z_lo, z_hi = z_lo - extra / 2, z_hi + extra / 2

    fig = plt.figure(figsize=(FIG_W, FIG_H))
    ax = fig.add_axes(AX_RECT)
    cax = fig.add_axes([AX_RECT[0] + AX_RECT[2] + 0.01, AX_RECT[1], 0.012, AX_RECT[3]])
    order = np.argsort(q)   # draw high charge on top
    m = ax.scatter(z[order], x[order], c=q[order], cmap="jet", vmin=0, vmax=np.percentile(q, 99), s=4, lw=0)
    fig.colorbar(m, cax=cax, label="charge")
    ax.plot(vtx[2], vtx[0], "*", ms=16, mfc="magenta", mec="k", mew=1, label=r"reco $\nu$ vertex", zorder=5)
    ax.legend(fontsize=8, loc="upper right", framealpha=0.8)
    for xb in TPC_X:
        if x_lo < xb < x_hi:
            ax.axhline(xb, c="k", lw=1.2, zorder=1)
    for zb in TPC_Z:
        if z_lo < zb < z_hi:
            ax.axvline(zb, c="k", lw=1.2, zorder=1)
    ax.set_xlim(z_lo, z_hi); ax.set_ylim(x_lo, x_hi)
    ax.set_aspect("equal", adjustable="box")
    ax.set(xlabel="z (cm)", ylabel="x (cm)")
    ax.set_title(f"Open data, {channel_latex} selection", fontsize=11)
    fig.text(AX_RECT[0] + AX_RECT[2] + 0.075, AX_RECT[1] + AX_RECT[3], info_label(row, channel),
             ha="left", va="top", fontsize=9, bbox=dict(boxstyle="round", fc="white", ec="0.6"))
    return fig

# Drawing The Displays

In [ ]:
saved = {}   # channel -> (savename stem, row)
for i, channel in enumerate(reco_categories):
    rows = chosen_df.filter(pl.col("display_channel") == channel).to_dicts()
    if not rows:
        continue
    row = rows[0]
    fig = draw_event(row, channel, reco_category_labels_latex[i])
    if fig is None:
        continue
    stem = f"{channel}_run{row['run']}_{row['subrun']}_{row['event']}"
    fig.savefig(f"{plots_dir}/{stem}.png")
    fig.savefig(f"{plots_dir}/{stem}.pdf")
    saved[channel] = (stem, row)
    if show_plots:
        plt.show()
    plt.close(fig)

# Whole-Detector Overlays

For each channel, up to `max_overlay_events` selected open-data events, overlaid in the whole TPC.

In [ ]:
def entries_for(df):
    # T_eval entry number of every row of df (matched by run/subrun/event within each file)
    entries = np.full(df.height, -1)
    for filename in df["filename"].unique():
        rse = _files.setdefault(filename, uproot.open(f"{data_files_location}/{filename}"))["wcpselection/T_eval"].arrays(
            ["run", "subrun", "event"], library="np")
        entry_of = {k: i for i, k in enumerate(zip(rse["run"], rse["subrun"], rse["event"]))}
        for row_i, (fn, r, s_, e) in enumerate(df.select(["filename", "run", "subrun", "event"]).iter_rows()):
            if fn == filename:
                entries[row_i] = entry_of[(r, s_, e)]
    assert (entries >= 0).all()
    return entries


# 20 distinct colors: the dark then the light half of tab20, so neighbouring events differ most
OVERLAY_COLORS = [plt.cm.tab20(k) for k in list(range(0, 20, 2)) + list(range(1, 20, 2))]
OV_PAD = 10.0   # cm around the TPC active volume
OV_Z = (TPC_Z[0] - OV_PAD, TPC_Z[1] + OV_PAD)
OV_X = (TPC_X[0] - OV_PAD, TPC_X[1] + OV_PAD)
OV_Y = (TPC_Y[0] - OV_PAD, TPC_Y[1] + OV_PAD)


def draw_overlay(sel, channel, channel_latex, n_selected):
    z_len, x_len, y_len = OV_Z[1] - OV_Z[0], OV_X[1] - OV_X[0], OV_Y[1] - OV_Y[0]
    width = 15.0
    fig = plt.figure(figsize=(width, width * (y_len + x_len) / (z_len + y_len) + 0.6))
    gs = fig.add_gridspec(2, 2, width_ratios=[z_len, y_len], height_ratios=[y_len, x_len],
                          left=0.05, right=0.99, bottom=0.08, top=0.98, wspace=0.03, hspace=0.04)
    ax_zx = fig.add_subplot(gs[1, 0])
    ax_zy = fig.add_subplot(gs[0, 0], sharex=ax_zx)
    ax_yx = fig.add_subplot(gs[1, 1], sharey=ax_zx)
    ax_info = fig.add_subplot(gs[0, 1]); ax_info.axis("off")

    for k, row in enumerate(sel.iter_rows(named=True)):
        x, y, z = read_spacepoints(row["filename"], row["entry"], coords=("x", "y", "z"))
        c = OVERLAY_COLORS[k % len(OVERLAY_COLORS)]
        for ax, h, v in [(ax_zx, z, x), (ax_zy, z, y), (ax_yx, y, x)]:
            ax.scatter(h, v, color=c, s=0.8, lw=0, rasterized=True)
        vx, vy, vz = row["wc_reco_nuvtxX"], row["wc_reco_nuvtxY"], row["wc_reco_nuvtxZ"]
        for ax, h, v in [(ax_zx, vz, vx), (ax_zy, vz, vy), (ax_yx, vy, vx)]:
            ax.plot(h, v, "o", ms=3, mfc=c, mec="k", mew=0.4, zorder=5)

    # TPC active volume
    for ax, (h0, h1), (v0, v1) in [(ax_zx, TPC_Z, TPC_X), (ax_zy, TPC_Z, TPC_Y), (ax_yx, TPC_Y, TPC_X)]:
        ax.add_patch(mpl.patches.Rectangle((h0, v0), h1 - h0, v1 - v0, fill=False, ec="k", lw=1))
        ax.set_aspect("equal", adjustable="box")
    ax_zx.set_xlim(*OV_Z); ax_zx.set_ylim(*OV_X)
    ax_zy.set_ylim(*OV_Y); ax_yx.set_xlim(*OV_Y)
    ax_zx.set(xlabel="z (cm)", ylabel="x (cm)")
    ax_zy.set(ylabel="y (cm)")
    ax_yx.set(xlabel="y (cm)")
    ax_zy.tick_params(labelbottom=False)
    ax_yx.tick_params(labelleft=False)
    shown = f"all {sel.height}" if sel.height == n_selected else f"{sel.height} random of {n_selected}"
    ax_info.text(0.05, 0.9, f"Open data\n{channel_latex} selection\n\n{shown}\nselected events", transform=ax_info.transAxes,
                 ha="left", va="top", fontsize=12)
    return fig


os.makedirs(f"{plots_dir}/overlays", exist_ok=True)
overlay_saved = {}   # channel -> (savename stem, number shown, number selected)
for i, channel in enumerate(reco_categories):
    sel = data_df.filter(pl.col("reco_channel") == channel).sort(["filename", "run", "subrun", "event"])
    n_selected = sel.height
    if n_selected == 0:
        continue
    if n_selected > max_overlay_events:
        rows = np.random.default_rng(overlay_seed).choice(n_selected, size=max_overlay_events, replace=False)
        sel = sel[np.sort(rows)]
    sel = sel.with_columns(pl.Series("entry", entries_for(sel)))
    fig = draw_overlay(sel, channel, reco_category_labels_latex[i], n_selected)
    stem = f"overlays/overlay_{channel}"
    fig.savefig(f"{plots_dir}/{stem}.png")
    fig.savefig(f"{plots_dir}/{stem}.pdf", dpi=200)
    overlay_saved[channel] = (stem, sel.height, n_selected)
    if show_plots:
        plt.show()
    plt.close(fig)

# LaTeX Output

Written as a subsection of the note (the note defines `\odedfigs` as the figure directory). One subsubsection per topology group; each channel gets
its example display followed by its whole-detector overlay.

In [ ]:
channel_groups = [
    ("Single-Photon Selections", ["1gNp", "1g0p", "1gNp1mu", "1g0p1mu", "1g_outFV"]),
    ("Two-Photon Selections", ["NC1pi0_Np", "NC1pi0_0p", "numuCC1pi0_Np", "numuCC1pi0_0p", "1pi0_outFV",
                               "multi_pi0", "eta_other", "pi0_dalitz_decay"]),
    (r"\texorpdfstring{$\nu_e$}{nue}CC Selections", ["nueCC_Np", "nueCC_0p"]),
    (r"\texorpdfstring{$\nu_\mu$}{numu}CC Selections", ["numuCC_Np", "numuCC_0p"]),
    ("Other Selections", ["other_outFV_dirt", "NC_no_gamma", "ext"]),
]
assert sorted(sum([g[1] for g in channel_groups], [])) == sorted(reco_categories)
latex_label = dict(zip(reco_categories, reco_category_labels_latex))

tex = [
    "\\subsection{Open Data Event Displays}",
    "\\label{sec:open_data_event_displays}",
    "",
    f"In this section, we show example event displays of runs 1-5 BNB open data events in each of the {len(reco_categories)} BAGELS selections. "
    "For each selection, we show one open data event passing that selection. "
    "Each display shows the Wire-Cell reconstructed 3D charge spacepoints of the neutrino-candidate cluster in the $x$-$z$ view "
    "($z$ along the beam, $x$ the drift coordinate), colored by charge. "
    "The window covers the 5th-95th percentile range of the spacepoints in each coordinate, expanded to include the reconstructed neutrino vertex (magenta star). "
    "Black lines mark the TPC active-volume boundaries where they fall inside the window. "
    "The box on the right lists reconstructed quantities and the BDT score for the selection.",
    "",
    "Each example display is followed by a view of the whole TPC with many open data events from the same selection overlaid. "
    f"These show the Wire-Cell reconstructed 3D charge spacepoints of up to {max_overlay_events} selected events, one color per event "
    "(colors repeat after 20 events). Selections with more events show a random subset. The main panel is the $z$-$x$ view, with the $z$-$y$ view "
    "above it and the $y$-$x$ view to its right. Each event's reconstructed neutrino vertex is a dot of its color, and the black boxes are the TPC "
    "active volume.",
    "",
]
for group_title, channels in channel_groups:
    tex += [f"\\subsubsection{{{group_title}}}", ""]
    for channel in channels:
        sentence = []
        if channel in saved:
            sentence.append(f"Figure~\\ref{{fig:oded_{channel}}} shows an example open data event passing the {latex_label[channel]} selection")
        if channel in overlay_saved:
            sentence.append(f"Figure~\\ref{{fig:oded_overlay_{channel}}} shows {'all' if overlay_saved[channel][1] == overlay_saved[channel][2] else 'many'} "
                            f"open data events passing it overlaid in the whole detector")
        if not sentence:
            continue
        tex += [", and ".join(sentence) + ".", ""]
        if channel in saved:
            stem, row = saved[channel]
            tex += ["\\begin{figure}[H]", "    \\centering",
                    f"    \\includegraphics[width=0.9\\textwidth]{{\\odedfigs/{stem}.pdf}}",
                    f"    \\caption{{Example runs 1-5 open data event display for the {latex_label[channel]} selection "
                    f"(run {row['run']}, subrun {row['subrun']}, event {row['event']}).}}",
                    f"    \\label{{fig:oded_{channel}}}",
                    "\\end{figure}", ""]
        if channel in overlay_saved:
            stem, n_shown, n_selected = overlay_saved[channel]
            shown = f"all {n_selected} selected events" if n_shown == n_selected else f"{n_shown} randomly chosen of the {n_selected} selected events"
            tex += ["\\begin{figure}[H]", "    \\centering",
                    f"    \\includegraphics[width=\\textwidth]{{\\odedfigs/{stem}.pdf}}",
                    f"    \\caption{{Runs 1-5 open data events in the {latex_label[channel]} selection, overlaid in the whole TPC ({shown}).}}",
                    f"    \\label{{fig:oded_overlay_{channel}}}",
                    "\\end{figure}", ""]

tex_path = f"{plots_dir}/open_data_event_displays.tex"
with open(tex_path, "w") as f:
    f.write("\n".join(tex))
print(f"wrote {tex_path}")